# Praktikum Visi Komputer — proses per sel

Kode algoritma ditampilkan langsung di bawah. Jalankan **Persiapan** sekali, lalu jalankan sel definisi fungsi yang diperlukan pada bagian yang ingin dicoba. Setiap sel operasi memproses satu contoh dan menampilkan hasilnya; tidak ada runner yang menjalankan semua operasi sekaligus.

Buka notebook dari root folder `viskom`. Gunakan kernel Python yang memiliki dependensi di `requirements.txt` (kernel Jupyter Viskom pada Suite Hub sudah disiapkan). Untuk instalasi pada environment lain: `%pip install -r requirements.txt`.

In [ ]:
from __future__ import annotations
from pathlib import Path
from collections import defaultdict
from dataclasses import dataclass
import csv, json, math, tempfile
import numpy as np
from PIL import Image as PILImage
from scipy import ndimage as ndi
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Ellipse, Rectangle
import cv2

ROOT = Path.cwd().resolve()
if not (ROOT / "hough_bab3.py").is_file():
    raise RuntimeError("Buka notebook dari root folder proyek viskom.")
FOLDER_CITRA = ROOT / "citra"
FOLDER_HASIL = Path(tempfile.mkdtemp(prefix="viskom_notebook_"))
print("Folder hasil sesi:", FOLDER_HASIL)


## 1. Hough Transform

Jalankan sel persiapan dan definisi di bagian ini terlebih dahulu. Setiap contoh memiliki sel pengolahan sendiri. Keempat citra ada di `viskom/citra`. Hasil USG dan perhitungan HC hanya untuk pembelajaran, bukan diagnosis klinis.

In [ ]:
KONFIGURASI = {
    "garis": {
        "gambar": "building.jpg",
        "ukuran": (868, 600),
        "roi": None,
        "jumlah": 10,
    },
    "lingkaran": {
        "gambar": "coins.png",
        "ukuran": (384, 303),
        "roi": None,
        "jumlah": 24,
        "rmin": 15,
        "rmax": 35,
    },
    "elips": {
        "gambar": "coffee.png",
        "ukuran": (600, 400),
        # x kiri, y atas, x kanan, y bawah.
        # Membatasi pencarian pada bibir cangkir.
        "roi": (165, 10, 425, 218),
        "iterasi": 20000,
    },
    "janin": {
        "gambar": "388_HC.png",
        "ukuran": (800, 540),
        "roi": (190, 190, 570, 440),
        "iterasi": 25000,
    },
}

### Persiapan citra dan deteksi tepi
Kode membaca gambar, mengubahnya ke grayscale, menghitung gradien, menipiskan tepi dan menerapkan ambang.

In [ ]:
def baca_citra(path):
    gambar = PILImage.open(path).convert("RGBA")

    # Bagian transparan dilapisi putih.
    latar = PILImage.new("RGBA", gambar.size, "white")
    latar.alpha_composite(gambar)

    return np.array(latar.convert("RGB"))

def deteksi_tepi(rgb, sigma=1.2):
    # RGB menjadi grayscale.
    gray = rgb.astype(float) @ np.array([0.299, 0.587, 0.114])

    # Mengurangi noise.
    smooth = ndi.gaussian_filter(gray, sigma)

    # Gradien horizontal dan vertikal.
    gx = ndi.sobel(smooth, axis=1)
    gy = ndi.sobel(smooth, axis=0)
    mag = np.hypot(gx, gy)

    yy, xx = np.indices(gray.shape)

    nx = gx / np.maximum(mag, 1e-9)
    ny = gy / np.maximum(mag, 1e-9)

    # Membandingkan gradien dengan tetangganya untuk menipiskan tepi.
    depan = ndi.map_coordinates(
        mag, [yy + ny, xx + nx], order=1
    )
    belakang = ndi.map_coordinates(
        mag, [yy - ny, xx - nx], order=1
    )

    tipis = (mag >= depan) & (mag >= belakang)

    # Ambang tepi kuat dan lemah.
    tinggi = max(30.0, 0.18 * float(mag.max()))

    kuat = tipis & (mag >= tinggi)
    lemah = tipis & (mag >= 0.5 * tinggi)

    # Mempertahankan tepi lemah yang terhubung dengan tepi kuat.
    edges = ndi.binary_propagation(kuat, mask=lemah)

    edges[[0, -1], :] = False
    edges[:, [0, -1]] = False

    return gray, edges, gx, gy

### Menyiapkan ROI dan visualisasi hasil
Fungsi di bawah dipakai oleh masing-masing contoh, tanpa menjalankan contoh lain.

In [ ]:
def siapkan_hough(mode):
    konfigurasi = KONFIGURASI[mode]
    path = FOLDER_CITRA / konfigurasi["gambar"]
    if not path.is_file():
        raise FileNotFoundError(f"Gambar tidak ditemukan: {path}")
    rgb = baca_citra(path)
    h, w = rgb.shape[:2]
    if (w, h) != konfigurasi["ukuran"]:
        raise ValueError(f"Ukuran {path.name}: {(w, h)}, diharapkan {konfigurasi['ukuran']}")
    _, edges, gx, gy = deteksi_tepi(rgb)
    roi = konfigurasi["roi"]
    if roi is not None:
        x1, y1, x2, y2 = roi
        if not (0 <= x1 < x2 <= w and 0 <= y1 < y2 <= h):
            raise ValueError("ROI berada di luar gambar")
        mask = np.zeros_like(edges, dtype=bool)
        mask[y1:y2, x1:x2] = True
        edges &= mask
    if not edges.any():
        raise ValueError("Peta tepi kosong")
    return path, rgb, edges, gx, gy, roi

def tampilkan_hasil(rgb, edges, mode, deteksi, roi, nama):
    h, w = edges.shape

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(rgb)
    axes[1].imshow(edges, cmap="gray", vmin=0, vmax=1)
    axes[2].imshow(rgb)

    axes[0].set_title("Citra asli dan area pencarian")
    axes[1].set_title(
        "Peta tepi dalam ROI" if roi else "Peta tepi"
    )
    axes[2].set_title(f"Hasil {mode}")

    # Kotak cyan memperlihatkan area pencarian.
    if roi:
        x1, y1, x2, y2 = roi

        axes[0].add_patch(Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            fill=False,
            edgecolor="cyan",
            linewidth=1.5,
            linestyle="--",
        ))

    for parameter in deteksi:
        if mode == "garis":
            t = math.radians(parameter["theta_derajat"])
            rho = parameter["rho"]

            x0 = rho * math.cos(t)
            y0 = rho * math.sin(t)
            panjang = math.hypot(h, w)

            axes[2].plot(
                [
                    x0 - panjang * math.sin(t),
                    x0 + panjang * math.sin(t),
                ],
                [
                    y0 + panjang * math.cos(t),
                    y0 - panjang * math.cos(t),
                ],
                color="red",
                linewidth=1,
            )

        elif mode == "lingkaran":
            axes[2].add_patch(Circle(
                (parameter["cx"], parameter["cy"]),
                parameter["r"],
                fill=False,
                color="red",
            ))

            axes[2].plot(
                parameter["cx"],
                parameter["cy"],
                "r+",
            )

        else:
            axes[2].add_patch(Ellipse(
                (parameter["cx"], parameter["cy"]),
                width=2 * parameter["a"],
                height=2 * parameter["b"],
                angle=parameter["sudut_derajat"],
                fill=False,
                color="lime",
                linewidth=2,
            ))

    for ax in axes:
        ax.set_xlim(-0.5, w - 0.5)
        ax.set_ylim(h - 0.5, -0.5)
        ax.axis("off")

    fig.tight_layout()

    lokasi = FOLDER_HASIL / f"{nama}_{mode}.png"
    fig.savefig(lokasi, dpi=150)

    plt.show()
    plt.close(fig)

### Hough garis — gedung

In [ ]:
def hough_garis(edges, jumlah=10):
    h, w = edges.shape
    batas = int(np.ceil(np.hypot(h, w)))

    theta = np.deg2rad(np.arange(-90, 90))
    acc = np.zeros((2 * batas + 1, len(theta)), np.int32)

    ys, xs = np.nonzero(edges)

    for k, t in enumerate(theta):
        rho = np.rint(
            xs * np.cos(t) + ys * np.sin(t)
        ).astype(int)

        np.add.at(acc[:, k], rho + batas, 1)

    work = acc.copy()
    hasil = []

    for _ in range(jumlah):
        ri, ti = np.unravel_index(
            np.argmax(work), work.shape
        )
        suara = int(work[ri, ti])

        if suara < 20:
            break

        hasil.append({
            "rho": int(ri - batas),
            "theta_derajat": int(ti - 90),
            "suara": suara,
        })

        # Menekan puncak tetangga agar tidak mendeteksi garis berulang.
        for dt in range(-6, 7):
            idx = ti + dt

            if 0 <= idx < len(theta):
                rr = ri
            else:
                rr = 2 * batas - ri

            work[
                max(0, rr - 10):rr + 11,
                idx % len(theta),
            ] = 0

    return hasil

In [ ]:
path_garis, rgb_garis, tepi_garis, gx_garis, gy_garis, roi_garis = siapkan_hough("garis")
garis = hough_garis(tepi_garis, jumlah=KONFIGURASI["garis"]["jumlah"])
print("Kandidat garis:", len(garis))
tampilkan_hasil(rgb_garis, tepi_garis, "garis", garis, roi_garis, path_garis.stem)

### Hough lingkaran — koin

In [ ]:
def hough_lingkaran(
    edges, gx, gy, rmin=15, rmax=35, jumlah=24
):
    h, w = edges.shape
    ys, xs = np.nonzero(edges)

    panjang = np.maximum(
        np.hypot(gx[ys, xs], gy[ys, xs]),
        1e-9,
    )

    nx = gx[ys, xs] / panjang
    ny = gy[ys, xs] / panjang

    radii = np.arange(rmin, rmax + 1)
    acc = np.zeros((len(radii), h, w), np.uint32)

    for k, r in enumerate(radii):
        for tanda in (-1, 1):
            cx = np.rint(xs + tanda * r * nx).astype(int)
            cy = np.rint(ys + tanda * r * ny).astype(int)

            valid = (
                (cx >= 0) & (cx < w)
                & (cy >= 0) & (cy < h)
            )

            np.add.at(
                acc[k],
                (cy[valid], cx[valid]),
                1,
            )

    smooth = ndi.uniform_filter(
        acc.astype(float),
        size=(3, 5, 5),
    )

    hasil = []
    yy, xx = np.ogrid[:h, :w]

    for _ in range(jumlah):
        k, cy, cx = np.unravel_index(
            np.argmax(smooth), smooth.shape
        )

        if smooth[k, cy, cx] <= 0.02:
            break

        r = int(radii[k])

        hasil.append({
            "cx": int(cx),
            "cy": int(cy),
            "r": r,
            "skor_akumulator": float(smooth[k, cy, cx]),
        })

        # Mencegah lingkaran ganda pada satu pusat.
        sekitar_pusat = (
            (xx - cx) ** 2 + (yy - cy) ** 2
            < max(20, r * 0.6) ** 2
        )

        smooth[:, sekitar_pusat] = 0

    return hasil

In [ ]:
path_koin, rgb_koin, tepi_koin, gx_koin, gy_koin, roi_koin = siapkan_hough("lingkaran")
aturan_koin = KONFIGURASI["lingkaran"]
lingkaran = hough_lingkaran(tepi_koin, gx_koin, gy_koin, rmin=aturan_koin["rmin"], rmax=aturan_koin["rmax"], jumlah=aturan_koin["jumlah"])
print("Kandidat lingkaran:", len(lingkaran))
tampilkan_hasil(rgb_koin, tepi_koin, "lingkaran", lingkaran, roi_koin, path_koin.stem)

### Hough elips — kopi
Fungsi berikut menghitung elips dari lima titik, menilai dukungan tepi, dan memilih kandidat dengan randomized Hough.

In [ ]:
def elips_dari_lima_titik(points):
    origin = points.mean(axis=0)
    scale = max(float(points.std()), 1.0)

    x, y = ((points - origin) / scale).T

    design = np.column_stack([
        x * x,
        x * y,
        y * y,
        x,
        y,
        np.ones(5),
    ])

    _, singular, vt = np.linalg.svd(
        design, full_matrices=True
    )

    if singular[-1] < 1e-7:
        return None

    A, B, C, D, E, F = vt[-1]

    # Syarat konik agar berbentuk elips.
    if B * B - 4 * A * C >= 0:
        return None

    Q = np.array([
        [A, B / 2],
        [B / 2, C],
    ])

    try:
        center = -0.5 * np.linalg.solve(Q, [D, E])

        value = (
            center @ Q @ center
            + np.dot([D, E], center)
            + F
        )

        eigen, vectors = np.linalg.eigh(Q)

        if np.any(np.abs(eigen) < 1e-12):
            return None

        axes2 = -value / eigen

        if np.any(axes2 <= 0):
            return None

        axes = np.sqrt(axes2) * scale
        j = int(np.argmax(axes))

        phi = math.atan2(
            vectors[1, j],
            vectors[0, j],
        ) % math.pi

        cx, cy = center * scale + origin

        return np.array([
            cx,
            cy,
            axes[j],
            axes[1 - j],
            phi,
        ])

    except np.linalg.LinAlgError:
        return None

def dukungan_elips(parameter, distance):
    cx, cy, a, b, phi = parameter

    t = np.linspace(
        0, 2 * math.pi, 360, endpoint=False
    )

    x = (
        cx
        + a * np.cos(t) * np.cos(phi)
        - b * np.sin(t) * np.sin(phi)
    )

    y = (
        cy
        + a * np.cos(t) * np.sin(phi)
        + b * np.sin(t) * np.cos(phi)
    )

    jarak = ndi.map_coordinates(
        distance,
        [y, x],
        order=1,
        mode="constant",
        cval=np.inf,
    )

    # Sampel dianggap didukung jika berjarak maksimal 2,5 piksel.
    return float(np.mean(jarak <= 2.5))

def hough_elips(edges, iterasi=20000, seed=42):
    ys, xs = np.nonzero(edges)
    points = np.column_stack([xs, ys]).astype(float)

    if len(points) < 5:
        return None

    h, w = edges.shape
    distance = ndi.distance_transform_edt(~edges)

    rng = np.random.default_rng(seed)
    bins = defaultdict(lambda: [0, None, 0.0])

    for _ in range(iterasi):
        indeks = rng.choice(len(points), 5, replace=False)
        parameter = elips_dari_lima_titik(points[indeks])

        if parameter is None:
            continue

        cx, cy, a, b, phi = parameter

        valid = (
            0 <= cx < w
            and 0 <= cy < h
            and 15 <= b <= a <= 0.55 * max(h, w)
            and b / a >= 0.35
        )

        if not valid:
            continue

        key = tuple(
            np.rint([
                cx / 8,
                cy / 8,
                a / 8,
                b / 8,
                np.rad2deg(phi) / 10,
            ]).astype(int)
        )

        score = dukungan_elips(parameter, distance)

        if score < 0.5:
            continue

        cell = bins[key]
        cell[0] += 1

        if score > cell[2]:
            cell[1] = parameter
            cell[2] = score

    if not bins:
        return None

    suara, parameter, score = max(
        bins.values(),
        key=lambda item: (item[0], item[2]),
    )

    return {
        "cx": float(parameter[0]),
        "cy": float(parameter[1]),
        "a": float(parameter[2]),
        "b": float(parameter[3]),
        "sudut_derajat": float(np.rad2deg(parameter[4])),
        "suara": int(suara),
        "dukungan_tepi": float(score),
    }

In [ ]:
path_kopi, rgb_kopi, tepi_kopi, gx_kopi, gy_kopi, roi_kopi = siapkan_hough("elips")
elips_kopi = hough_elips(tepi_kopi, iterasi=KONFIGURASI["elips"]["iterasi"], seed=42)
print("Kandidat elips:", elips_kopi)
tampilkan_hasil(rgb_kopi, tepi_kopi, "elips", [] if elips_kopi is None else [elips_kopi], roi_kopi, path_kopi.stem)

### Hough janin — citra HC18
Perhitungan memakai metadata khusus `388_HC.png`. Jangan gunakan metadata itu untuk gambar lain.

In [ ]:
def baca_metadata(gambar):
    lokasi = ROOT / "metadata_hc18.csv"

    if not lokasi.is_file():
        raise FileNotFoundError(
            f"Metadata tidak ditemukan: {lokasi}"
        )

    with lokasi.open(
        newline="", encoding="utf-8-sig"
    ) as file:
        for row in csv.DictReader(file):
            row = {
                k.strip().lower(): v
                for k, v in row.items()
                if k
            }

            if row.get("filename", "").strip() == gambar.name:
                skala = float(row["pixel size(mm)"])
                acuan = float(row["head circumference (mm)"])

                if (
                    not math.isfinite(skala)
                    or not math.isfinite(acuan)
                    or skala <= 0
                    or acuan <= 0
                ):
                    raise ValueError(
                        "Skala dan HC acuan harus positif."
                    )

                return skala, acuan

    raise ValueError(
        f"Metadata {gambar.name} tidak ditemukan."
    )

def hitung_biometri(elips, skala, hc_acuan):
    a = elips["a"]
    b = elips["b"]

    # a dan b adalah setengah panjang sumbu.
    hc_px = math.pi * (
        3 * (a + b)
        - math.sqrt((3 * a + b) * (a + 3 * b))
    )

    hc_mm = hc_px * skala

    hasil = {
        "hc_px": hc_px,
        "hc_mm": hc_mm,
        "hc_acuan_mm": hc_acuan,
        "selisih_hc_mm": abs(hc_mm - hc_acuan),
    }

    # Demonstrasi proksi BPD, bukan pengukuran BPD klinis.
    bpd_cm = 2 * b * skala / 10
    hasil["proksi_bpd_cm"] = bpd_cm
    hasil["catatan"] = (
        "Demonstrasi geometris; bukan pengukuran klinis."
    )

    if 1.4 <= bpd_cm <= 10.17:
        ga = (
            9.54
            + 1.482 * bpd_cm
            + 0.1676 * bpd_cm ** 2
        )

        total_hari = int(round(ga * 7))

        hasil["ga_minggu"] = ga
        hasil["minggu"] = total_hari // 7
        hasil["hari"] = total_hari % 7

    return hasil

In [ ]:
path_janin, rgb_janin, tepi_janin, gx_janin, gy_janin, roi_janin = siapkan_hough("janin")
elips_janin = hough_elips(tepi_janin, iterasi=KONFIGURASI["janin"]["iterasi"], seed=42)
if elips_janin is None:
    print("Tidak ada kandidat elips yang lolos; periksa ROI dan parameter.")
else:
    skala_mm, hc_acuan_mm = baca_metadata(path_janin)
    print("Elips:", elips_janin)
    print("Biometri demonstrasi:", hitung_biometri(elips_janin, skala_mm, hc_acuan_mm))
tampilkan_hasil(rgb_janin, tepi_janin, "janin", [] if elips_janin is None else [elips_janin], roi_janin, path_janin.stem)

## 2. Dasar pengolahan citra digital

Jalankan sel persiapan bagian ini, lalu sel operasi yang ingin dicoba. Perhitungan berikut menggunakan matriks piksel dan rumus proyek.

In [ ]:
@dataclass
class Image:
    width: int
    height: int
    pixels: list[list[tuple[int, int, int]]]
    mode: str = "RGB"

    def __post_init__(self):
        if self.width <= 0 or self.height <= 0:
            raise ValueError("Image dimensions must be positive")
        if len(self.pixels) != self.height or any(len(row) != self.width for row in self.pixels):
            raise ValueError("Pixel matrix dimensions do not match image dimensions")

def _clamp(value):
    return max(0, min(255, round(value)))

def read_image(filename):
    from PIL import Image as Decoder
    with Decoder.open(filename) as decoded:
        mode = "L" if decoded.mode == "L" else "RGB"
        rgb = decoded.convert("RGB")
        width, height = rgb.size
        flat = list(rgb.get_flattened_data() if hasattr(rgb, "get_flattened_data") else rgb.getdata())
    return Image(width, height, [flat[y*width:(y+1)*width] for y in range(height)], mode)

def bandingkan_manual(sebelum, sesudah, judul):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, gambar, label in zip(axes, (sebelum, sesudah), ("Sebelum", judul)):
        ax.imshow(np.array(gambar.pixels, dtype=np.uint8))
        ax.set_title(label)
        ax.axis("off")
    plt.show()

dasar_input = ROOT / "dasar_pengolahan_citra_digital/data/input"
citra_dasar = read_image(dasar_input / "sample_asymmetric.png")

### Properti citra

In [ ]:
def properties(image):
    channels=1 if image.mode=='L' else 3
    pixels=image.width*image.height
    return {'width':image.width,'height':image.height,'pixels':pixels,
            'megapixels':pixels/1_000_000,'mode':image.mode,'channels':channels,
            'bits_per_channel':8,'bits_per_pixel':8*channels,
            'levels_per_channel':256,'color_combinations':2**(8*channels),
            'raw_bytes':pixels*channels}

print(json.dumps(properties(citra_dasar), indent=2))

### Kedalaman warna

In [ ]:
def color_depth(image, bits):
    if type(bits) is not int or not 1<=bits<=8: raise ValueError('bits must be 1..8')
    levels=(1<<bits)-1
    def q(v): return round(round(v/255*levels)/levels*255)
    return Image(image.width,image.height,[[tuple(q(v) for v in p) for p in row] for row in image.pixels],image.mode)

hasil_depth = color_depth(citra_dasar, 4)
bandingkan_manual(citra_dasar, hasil_depth, "Kuantisasi 4 bit")

#### Kedalaman warna 1 bit

In [ ]:
hasil_depth_1 = color_depth(citra_dasar, 1)
bandingkan_manual(citra_dasar, hasil_depth_1, "Kuantisasi 1 bit")

#### Kedalaman warna 8 bit

In [ ]:
hasil_depth_8 = color_depth(citra_dasar, 8)
bandingkan_manual(citra_dasar, hasil_depth_8, "Kuantisasi 8 bit")

### Resolusi

In [ ]:
def resize_nearest(image,width,height):
    if type(width) is not int or type(height) is not int or width<=0 or height<=0: raise ValueError('Dimensions must be positive')
    return Image(width,height,[[image.pixels[min(image.height-1,y*image.height//height)][min(image.width-1,x*image.width//width)] for x in range(width)] for y in range(height)],image.mode)

hasil_resize = resize_nearest(citra_dasar, 160, 100)
bandingkan_manual(citra_dasar, hasil_resize, "Resize 160 × 100")

### Rotasi

In [ ]:
def rotate(image,angle):
    if type(angle) is not int: raise ValueError('angle must be an integer')
    angle%=360
    if angle not in (0,90,180,270): raise ValueError('angle must be 0, 90, 180 or 270')
    if angle==0: rows=[row[:] for row in image.pixels]
    elif angle==90: rows=[[image.pixels[y][x] for y in range(image.height-1,-1,-1)] for x in range(image.width)]
    elif angle==180: rows=[row[::-1] for row in image.pixels[::-1]]
    else: rows=[[image.pixels[y][x] for y in range(image.height)] for x in range(image.width-1,-1,-1)]
    return Image(len(rows[0]),len(rows),rows,image.mode)

hasil_rotasi = rotate(citra_dasar, 90)
bandingkan_manual(citra_dasar, hasil_rotasi, "Rotasi 90°")

#### Rotasi 180°

In [ ]:
hasil_rotasi_180 = rotate(citra_dasar, 180)
bandingkan_manual(citra_dasar, hasil_rotasi_180, "Rotasi 180°")

#### Rotasi 270°

In [ ]:
hasil_rotasi_270 = rotate(citra_dasar, 270)
bandingkan_manual(citra_dasar, hasil_rotasi_270, "Rotasi 270°")

### Fungsi pendukung filter
Sel ini mendefinisikan akses piksel, grayscale, kernel Gaussian, konvolusi dan gradien yang dipakai beberapa proses berikut.

In [ ]:
def _pixel(image: Image, x: int, y: int) -> tuple[int,int,int]:
    return image.pixels[max(0, min(image.height - 1, y))][max(0, min(image.width - 1, x))]

def _gray(image: Image) -> list[list[float]]:
    return [[0.299*r+0.587*g+0.114*b for r,g,b in row] for row in image.pixels]

def _from_gray(values: list[list[float]]) -> Image:
    return Image(len(values[0]),len(values),[[( _clamp(value),_clamp(value),_clamp(value)) for value in row] for row in values])

def _kernel_gaussian(size: int, sigma: float|None=None) -> list[list[float]]:
    if size<3 or size%2==0: raise ValueError("kernel must be an odd integer >= 3")
    sigma=sigma or max(0.8,size/3); radius=size//2; kernel=[]; total=0.0
    for y in range(-radius,radius+1):
        row=[]
        for x in range(-radius,radius+1):
            value=math.exp(-(x*x+y*y)/(2*sigma*sigma)); row.append(value); total+=value
        kernel.append(row)
    return [[value/total for value in row] for row in kernel]

def _convolve(values: list[list[float]], kernel: list[list[float]]) -> list[list[float]]:
    height=len(values); width=len(values[0]); radius=len(kernel)//2; output=[]
    for y in range(height):
        row=[]
        for x in range(width):
            total=0.0
            for ky in range(len(kernel)):
                for kx in range(len(kernel)):
                    sy=max(0,min(height-1,y+ky-radius)); sx=max(0,min(width-1,x+kx-radius)); total+=values[sy][sx]*kernel[ky][kx]
            row.append(total)
        output.append(row)
    return output

def _filter_channels(image: Image, kernel: list[list[float]]) -> Image:
    channels=[]
    for channel in range(3):
        values=[[float(image.pixels[y][x][channel]) for x in range(image.width)] for y in range(image.height)]
        channels.append(_convolve(values,kernel))
    rows=[]
    for y in range(image.height):
        row=[]
        for x in range(image.width):
            row.append((_clamp(channels[0][y][x]), _clamp(channels[1][y][x]), _clamp(channels[2][y][x])))
        rows.append(row)
    return Image(image.width,image.height,rows)

def _sobel(values: list[list[float]]) -> tuple[list[list[float]],list[list[float]]]:
    gx_kernel=((-1,0,1),(-2,0,2),(-1,0,1)); gy_kernel=((-1,-2,-1),(0,0,0),(1,2,1)); height=len(values); width=len(values[0]); gx=[]; gy=[]
    for y in range(height):
        row_x=[]; row_y=[]
        for x in range(width):
            total_x=total_y=0.0
            for ky in range(3):
                for kx in range(3):
                    value=values[max(0,min(height-1,y+ky-1))][max(0,min(width-1,x+kx-1))]; total_x+=value*gx_kernel[ky][kx]; total_y+=value*gy_kernel[ky][kx]
            row_x.append(total_x); row_y.append(total_y)
        gx.append(row_x); gy.append(row_y)
    return gx,gy

### Pencerminan

In [ ]:
def mirror(image: Image, direction: str) -> Image:
    if direction not in ("horizontal","vertical"): raise ValueError("direction must be horizontal or vertical")
    rows=[list(reversed(row)) for row in image.pixels] if direction=="horizontal" else list(reversed(image.pixels))
    return Image(image.width,image.height,rows)

gambar_mirror = read_image(dasar_input / "internet_apple.png")
bandingkan_manual(gambar_mirror, mirror(gambar_mirror, "horizontal"), "Mirror horizontal")

#### Pencerminan vertikal

In [ ]:
gambar_mirror_vertikal = read_image(dasar_input / "internet_apple.png")
bandingkan_manual(gambar_mirror_vertikal, mirror(gambar_mirror_vertikal, "vertical"), "Mirror vertikal")

### Deteksi tepi Canny manual

In [ ]:
def edge(image: Image, low: int, high: int, blur: int) -> Image:
    if low<0 or high<=low: raise ValueError("high must be greater than low >= 0")
    blurred=_convolve(_gray(image),_kernel_gaussian(blur)); gx,gy=_sobel(blurred); height=len(blurred); width=len(blurred[0])
    magnitude=[[math.hypot(gx[y][x],gy[y][x]) for x in range(width)] for y in range(height)]; angle=[[math.degrees(math.atan2(gy[y][x],gx[y][x]))%180 for x in range(width)] for y in range(height)]; thin=[[0.0]*width for _ in range(height)]
    for y in range(1,height-1):
        for x in range(1,width-1):
            direction=angle[y][x]
            if direction<22.5 or direction>=157.5: a,b=magnitude[y][x-1],magnitude[y][x+1]
            elif direction<67.5: a,b=magnitude[y-1][x-1],magnitude[y+1][x+1]
            elif direction<112.5: a,b=magnitude[y-1][x],magnitude[y+1][x]
            else: a,b=magnitude[y-1][x+1],magnitude[y+1][x-1]
            if magnitude[y][x]>=a and magnitude[y][x]>=b: thin[y][x]=magnitude[y][x]
    strong=255.0; weak=75.0; classified=[[strong if value>=high else weak if value>=low else 0.0 for value in row] for row in thin]; changed=True
    while changed:
        changed=False
        for y in range(1,height-1):
            for x in range(1,width-1):
                if classified[y][x]==weak and any(classified[ny][nx]==strong for ny in range(y-1,y+2) for nx in range(x-1,x+2)): classified[y][x]=strong; changed=True
    return _from_gray([[strong if value==strong else 0.0 for value in row] for row in classified])

gambar_tepi = read_image(dasar_input / "edge_coins.png")
bandingkan_manual(gambar_tepi, edge(gambar_tepi, 40, 100, 5), "Canny manual")

### Reduksi noise
Fungsi sama digunakan pada tiga sel terpisah untuk median, Gaussian, dan bilateral.

In [ ]:
def denoise(image: Image, method: str, kernel: int) -> Image:
    if method not in ("gaussian","median","bilateral"): raise ValueError("Unknown denoise method")
    if kernel<3 or kernel%2==0: raise ValueError("kernel must be an odd integer >= 3")
    if method=="gaussian": return _filter_channels(image,_kernel_gaussian(kernel))
    radius=kernel//2; output=[]
    for y in range(image.height):
        row=[]
        for x in range(image.width):
            samples=[_pixel(image,sx,sy) for sy in range(y-radius,y+radius+1) for sx in range(x-radius,x+radius+1)]
            if method=="median": row.append(tuple(sorted(pixel[c] for pixel in samples)[len(samples)//2] for c in range(3))); continue
            sigma_space=max(1.0,kernel/2); sigma_range=50.0; center=_pixel(image,x,y); weighted=[0.0,0.0,0.0]; total=0.0
            for sy in range(y-radius,y+radius+1):
                for sx in range(x-radius,x+radius+1):
                    pixel=_pixel(image,sx,sy); spatial=math.exp(-((sx-x)**2+(sy-y)**2)/(2*sigma_space*sigma_space)); difference=sum((pixel[c]-center[c])**2 for c in range(3)); weight=spatial*math.exp(-difference/(2*sigma_range*sigma_range)); total+=weight
                    for c in range(3): weighted[c]+=pixel[c]*weight
            row.append(tuple(_clamp(value/total) for value in weighted))
        output.append(row)
    return Image(image.width,image.height,output)

#### Filter Median

In [ ]:
gambar_median = read_image(dasar_input / "median_input.png")
hasil_median = denoise(gambar_median, "median", 3)
bandingkan_manual(gambar_median, hasil_median, "Median kernel 3")

#### Filter Gaussian

In [ ]:
gambar_gaussian = read_image(dasar_input / "gaussian_input.png")
hasil_gaussian = denoise(gambar_gaussian, "gaussian", 3)
bandingkan_manual(gambar_gaussian, hasil_gaussian, "Gaussian kernel 3")

#### Filter Bilateral

In [ ]:
gambar_bilateral = read_image(dasar_input / "bilateral_input.png")
hasil_bilateral = denoise(gambar_bilateral, "bilateral", 5)
bandingkan_manual(gambar_bilateral, hasil_bilateral, "Bilateral kernel 5")

### Brightness

In [ ]:
def brightness(image: Image, c: int) -> Image:
    return Image(image.width,image.height,[[( _clamp(r+c),_clamp(g+c),_clamp(b+c)) for r,g,b in row] for row in image.pixels])

gambar_intensitas = read_image(dasar_input / "brightness_contrast.jpeg")
bandingkan_manual(gambar_intensitas, brightness(gambar_intensitas, 40), "Brightness +40")

### Contrast

In [ ]:
def contrast(image: Image, g: float, p: float=127) -> Image:
    return Image(image.width,image.height,[[( _clamp(g*r+p*(1-g)),_clamp(g*green+p*(1-g)),_clamp(g*b+p*(1-g))) for r,green,b in row] for row in image.pixels])

gambar_kontras = read_image(dasar_input / "brightness_contrast.jpeg")
bandingkan_manual(gambar_kontras, contrast(gambar_kontras, 1.5, 127), "Contrast 1.5")

### Fitur warna HSV
Momen warna dan histogram dihitung dari gambar fitur warna. Kode ini merupakan versi sel dari `fitur_warna.py`.

In [ ]:
path_warna = dasar_input / "fitur_warna.jpg"
citra_warna = cv2.imread(str(path_warna))
if citra_warna is None:
    raise FileNotFoundError(f"Citra tidak terbaca: {path_warna}")
hsv = cv2.cvtColor(citra_warna, cv2.COLOR_BGR2HSV)
piksel = hsv.reshape(-1, 3).astype(float)
fitur_momen = np.concatenate([piksel.mean(axis=0), piksel.std(axis=0)])
print("Momen HSV (mean, std):", np.round(fitur_momen, 2))
plt.imshow(cv2.cvtColor(citra_warna, cv2.COLOR_BGR2RGB))
plt.title("Citra fitur warna"); plt.axis("off"); plt.show()

#### Histogram warna dan vektor fitur

In [ ]:
fitur_hist = []
for kanal, rentang in zip(range(3), [180, 256, 256]):
    hist = cv2.calcHist([hsv], [kanal], None, [8], [0, rentang])
    hist = hist / hist.sum()
    fitur_hist.extend(hist.flatten())
fitur_hist = np.array(fitur_hist)
fitur_warna = np.concatenate([fitur_momen, fitur_hist])
print("Histogram HSV:", np.round(fitur_hist, 3))
print("Ukuran vektor fitur:", fitur_warna.shape)

## 3. Morfologi citra

Jalankan sel persiapan. Setiap operasi menggunakan citra biner yang sama. Hit-or-miss terlebih dahulu menghitung thinning agar dapat mencari endpoint.

In [ ]:
def load_grayscale(path: str | Path) -> np.ndarray:
    """Load a PNG (or another OpenCV-supported image) as one grayscale plane."""
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f"Cannot read image: {path}")
    if image.ndim != 2 or image.size == 0:
        raise ValueError(f"Input must be a non-empty grayscale image: {path}")
    return image

def binarize_document(gray: np.ndarray) -> np.ndarray:
    """Convert dark document ink to white foreground on a black background."""
    if gray.ndim != 2:
        raise ValueError("gray must be a single-channel image")
    _threshold, binary = cv2.threshold(
        gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )
    return binary

def morphology_kernel(size: int = 3) -> np.ndarray:
    """Return a cross-shaped structuring element for document strokes."""
    if size < 3 or size % 2 == 0:
        raise ValueError("kernel size must be an odd integer >= 3")
    return cv2.getStructuringElement(cv2.MORPH_CROSS, (size, size))

def tampil_morfologi(sebelum, sesudah, judul):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, gambar, label in zip(axes, (sebelum, sesudah), ("Input biner", judul)):
        ax.imshow(gambar, cmap="gray", vmin=0, vmax=255)
        ax.set_title(label)
        ax.axis("off")
    plt.show()

gray_document = load_grayscale(ROOT / "morfologi_citra/data/input/document_scan.png")
binary_document = binarize_document(gray_document)
kernel = morphology_kernel(3)
tampil_morfologi(gray_document, binary_document, "Biner Otsu inverse")

### Dilation

In [ ]:
def dilation(binary: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    """Thicken foreground strokes and bridge small gaps."""
    return cv2.dilate(binary, kernel, iterations=1)

hasil_dilation = dilation(binary_document, kernel)
tampil_morfologi(binary_document, hasil_dilation, "Dilation")

### Erosion

In [ ]:
def erosion(binary: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    """Thin foreground strokes and remove structures smaller than the kernel."""
    return cv2.erode(binary, kernel, iterations=1)

hasil_erosion = erosion(binary_document, kernel)
tampil_morfologi(binary_document, hasil_erosion, "Erosion")

### Opening

In [ ]:
def opening(binary: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    """Erosion followed by dilation; useful for removing small foreground noise."""
    return cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

hasil_opening = opening(binary_document, kernel)
tampil_morfologi(binary_document, hasil_opening, "Opening")

### Closing

In [ ]:
def closing(binary: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    """Dilation followed by erosion; useful for closing small gaps and holes."""
    return cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

hasil_closing = closing(binary_document, kernel)
tampil_morfologi(binary_document, hasil_closing, "Closing")

### Top-hat

In [ ]:
def top_hat(binary: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    """Extract small bright foreground details removed by opening.

    With this pipeline, foreground ink is represented by white pixels. Thus
    top-hat highlights small white components that are not preserved by the
    opening operation.
    """
    return cv2.morphologyEx(binary, cv2.MORPH_TOPHAT, kernel)

hasil_top_hat = top_hat(binary_document, kernel)
tampil_morfologi(binary_document, hasil_top_hat, "Top-hat")

### Skeletonization

In [ ]:
def skeletonization(binary: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    """Compute a morphological skeleton by iterative erosion and opening."""
    current = binary.copy()
    skeleton = np.zeros_like(binary)
    while cv2.countNonZero(current) > 0:
        opened = cv2.morphologyEx(current, cv2.MORPH_OPEN, kernel)
        residue = cv2.subtract(current, opened)
        skeleton = cv2.bitwise_or(skeleton, residue)
        current = cv2.erode(current, kernel, iterations=1)
    return skeleton

hasil_skeletonization = skeletonization(binary_document, kernel)
tampil_morfologi(binary_document, hasil_skeletonization, "Skeletonization")

### Thinning (Zhang-Suen)

In [ ]:
def _zhang_suen_delete_mask(image: np.ndarray, phase: int) -> np.ndarray:
    """Return pixels deleted by one Zhang-Suen thinning sub-iteration."""
    padded = np.pad(image, 1, mode="constant", constant_values=0)
    p2 = padded[:-2, 1:-1]  # north
    p3 = padded[:-2, 2:]  # north-east
    p4 = padded[1:-1, 2:]  # east
    p5 = padded[2:, 2:]  # south-east
    p6 = padded[2:, 1:-1]  # south
    p7 = padded[2:, :-2]  # south-west
    p8 = padded[1:-1, :-2]  # west
    p9 = padded[:-2, :-2]  # north-west
    neighbors = (p2, p3, p4, p5, p6, p7, p8, p9)

    neighbor_count = sum(neighbors)
    transitions = sum(
        ((left == 0) & (right == 1)).astype(np.uint8)
        for left, right in zip(neighbors, neighbors[1:] + neighbors[:1])
    )
    foreground = image == 1
    conditions = foreground & (neighbor_count >= 2) & (neighbor_count <= 6)
    conditions &= transitions == 1

    if phase == 1:
        conditions &= (p2 * p4 * p6) == 0
        conditions &= (p4 * p6 * p8) == 0
    else:
        conditions &= (p2 * p4 * p8) == 0
        conditions &= (p2 * p6 * p8) == 0
    return conditions

def thinning(binary: np.ndarray, max_iterations: int = 1000) -> np.ndarray:
    """Thin foreground strokes using the Zhang-Suen algorithm."""
    image = (binary > 0).astype(np.uint8)
    for _iteration in range(max_iterations):
        changed = False
        for phase in (1, 2):
            delete = _zhang_suen_delete_mask(image, phase)
            if np.any(delete):
                image[delete] = 0
                changed = True
        if not changed:
            return image * 255
    raise RuntimeError("Zhang-Suen thinning did not converge within the iteration limit")

In [ ]:
hasil_thinning = thinning(binary_document)
tampil_morfologi(binary_document, hasil_thinning, "Thinning")

### Hit-or-miss untuk endpoint

In [ ]:
def _endpoint_kernels() -> list[np.ndarray]:
    """Build eight hit-or-miss kernels for one-pixel line endpoints."""
    kernels: list[np.ndarray] = []
    for row in range(3):
        for column in range(3):
            if (row, column) == (1, 1):
                continue
            pattern = np.full((3, 3), -1, dtype=np.int8)
            pattern[1, 1] = 1
            pattern[row, column] = 1
            kernels.append(pattern)
    return kernels

def hit_or_miss(binary: np.ndarray) -> np.ndarray:
    """Mark endpoints in a thinned binary image using hit-or-miss patterns."""
    source = (binary > 0).astype(np.uint8)
    result = np.zeros_like(source)
    for pattern in _endpoint_kernels():
        matches = cv2.morphologyEx(source, cv2.MORPH_HITMISS, pattern)
        result = cv2.bitwise_or(result, matches)
    return result * 255

In [ ]:
hasil_endpoint = hit_or_miss(thinning(binary_document))
tampil_morfologi(binary_document, hasil_endpoint, "Hit-or-miss endpoint")